In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
root_dir = "/content/drive/MyDrive/RAG_Project/RAG_1"
config_dir = "/content/drive/MyDrive/RAG_Project/RAG_1/config"
data_dir = "/content/drive/MyDrive/RAG_Project/RAG_1/data"

In [6]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.0/349.0 kB 1.4 MB/s eta 0:00:00


In [ ]:
import os
requirement_file_path = os.path.join(root_dir, 'requirements.txt')
with open(requirement_file_path, 'r') as f:
    f.write("""

    """)

print(f"Created requirement file at: {requirement_file_path}")

In [3]:
import os

config_file_path = os.path.join(config_dir, 'config.yaml')

# Create an empty config.yaml file with correct YAML format
with open(config_file_path, 'w') as f:
    f.write("""
data_paths:
    raw_dir: "./data/Documention"
    processed_dir: "./data/processed"
vector_db:
    local_path: "./data/processed/chroma_db"  # Nếu dùng Vector DB dạng file như ChromaDB/FAISS
""")

print(f"Created config file at: {config_file_path}")

Created config file at: /content/drive/MyDrive/RAG_Project/RAG_1/config/config.yaml


In [4]:
import os
src_dir = "/content/drive/MyDrive/RAG_Project/RAG_1/src"
def create_python_file(python_file_name, text):
    os.makedirs(src_dir, exist_ok=True)

    if not python_file_name.endswith(".py"):
        python_file_name += ".py"

    python_file = os.path.join(src_dir, python_file_name)

    with open(python_file, "w", encoding="utf-8") as f:
        f.write(text)

    print(f"Created python file: {python_file}")
    return python_file

In [32]:
text = """from pypdf import PdfReader
def load_pdf_from_page(pdf_path, start_page=14):
    " start_page=16 vì PdfReader đánh số từ 0 PDF page 14 => index 14 "

    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages[start_page:]:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\\n"

    return text
"""
python_file_name = "load_pdf.py"
create_python_file(python_file_name, text)

Created python file: /content/drive/MyDrive/RAG_Project/RAG_1/src/load_pdf.py


'/content/drive/MyDrive/RAG_Project/RAG_1/src/load_pdf.py'

## Bước 1: Đọc file PDF

In [44]:
import sys
import importlib

# Check if 'load_pdf' module is already in sys.modules and reload it if necessary
if 'load_pdf' in sys.modules:
    importlib.reload(sys.modules['load_pdf'])

%cd /content/drive/MyDrive/RAG_Project/RAG_1/src/
from load_pdf import load_pdf_from_page
text = load_pdf_from_page(
    "/content/drive/MyDrive/RAG_Project/RAG_1/data/Documention/Giao_Trinh_KienTrucIoT.pdf"
)
print(text[:1000])

/content/drive/MyDrive/RAG_Project/RAG_1/src
Chương 1. Kiến trúc IoT  
 
2 
 
CHƯƠNG 1. KIẾN TRÚC IoT  
 Chương 1 cung cấp cái nhìn tổng quan về IoT, bao gồm các khái niệm cơ 
bản của nó. Chương này cũng bao g ồm các khái ni ệm và ki ến trúc. Ngoài ra, 
chương bao hàm ph ạm vi r ộng nhiều khía c ạnh kỹ thuật cũng như phi k ỹ thuật 
bằng cách trình bày một khuôn khổ bốn lớp đề cập đến các công nghệ cho phép 
thực hiện IoT, các tính chất bắt nguồn từ các hệ thống CNTT-TT hiện đại và cách 
chúng đang hỗ trợ IoT, tiềm năng cũng như các cải tiến mới dựa trên hệ sinh thái 
IoT và cuối cùng là các tác động và thách thức của IoT. 
1.1 Tổng quan về IoT  
Đầu năm 1926, Nikola Tesla đã hình dung ra m ột “thế giới được kết nối”. 
Ông nói với Tạp chí Colliers trong một cuộc phỏng vấn [84]: 
“Khi không dây được áp dụng một cách hoàn hảo, toàn bộ Trái đất sẽ 
được chuyển đổi thành một bộ não khổng lồ, trên thực tế, tất cả mọi thứ đều 
là những hạt của một tổng thể thực sự và nhịp nhàng … và các công 

## Bước 2: Chunking

In [40]:
text = r'''from pypdf import PdfReader

def chunk_text(text, chunk_size=1000, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks
def recursive_chunk_text(text, chunk_size):
    # Phương pháp 2: Recursive Character Chunking
    # Các ký tự phân tách ưu tiên: Chương -> Mục -> Xuống dòng -> Khoảng trắng
    separators = ["\n# ", "\n## ", "\n### ", "\n", " ", ""]
    return split_recursive(text, chunk_size, separators)
def split_recursive(input_text, chunk_size, separators):
    if len(input_text) <= chunk_size:
        return [input_text]
    if not separators:
        result = []
        for i in range(0, len(input_text), chunk_size):
            result.append(input_text[i:i + chunk_size])
        return result

    separator = separators[0]
    pieces = input_text.split(separator)
    chunks = []
    current = ""
    for piece in pieces:
        candidate = current + separator + piece
        if len(candidate) < chunk_size:
            current = candidate
        else:
            if current:
                chunks.append(current)
            current = piece
    if current:
        chunks.append(current)
    final_chunks = []
    for chunk in chunks:
        if len(chunk) > chunk_size:
            final_chunks.extend(split_recursive(chunk, chunk_size, separators[1:]))
        else:
            final_chunks.append(chunk)
    return final_chunks
'''
python_file_name = "chunking.py"
create_python_file(python_file_name, text)

Created python file: /content/drive/MyDrive/RAG_Project/RAG_1/src/chunking.py


'/content/drive/MyDrive/RAG_Project/RAG_1/src/chunking.py'

In [86]:
# Check if 'load_pdf' module is already in sys.modules and reload it if necessary
if 'chunking' in sys.modules:
    importlib.reload(sys.modules['chunking'])

%cd /content/drive/MyDrive/RAG_Project/RAG_1/src/
from chunking import chunk_text, recursive_chunk_text
fixed_chunks = chunk_text(text,chunk_size=5000)

recursive_chunks = recursive_chunk_text(text,chunk_size=5000)
print("Fixed:", len(fixed_chunks))
print("Recursive:", len(recursive_chunks))

/content/drive/MyDrive/RAG_Project/RAG_1/src
Fixed: 75
Recursive: 74


In [55]:
def statistics(chunks):

    lengths = [ len(c) for c in chunks ]

    print(f"Số chunks: {len(chunks)}")
    print(f"Min: {min(lengths)}")
    print(f"Max: {max(lengths)}")
    print(f"Avg: {sum(lengths)/len(lengths):.2f}")

In [87]:
print("\nFIXED")
statistics(fixed_chunks)

print("\nRECURSIVE")
statistics(recursive_chunks)


FIXED
Số chunks: 75
Min: 4345
Max: 5000
Avg: 4991.27

RECURSIVE
Số chunks: 74
Min: 4862
Max: 4999
Avg: 4957.74


In [88]:
import json
# lưu chunk vào file JSON chưa động đến db
data_dir = "/content/drive/MyDrive/RAG_Project/RAG_1/data"
processed_dir = data_dir + "/processed"
fixed_chunk_file_path = os.path.join(processed_dir, 'fixed.json')
recursive_chunk_file_path = os.path.join(processed_dir, 'recursive.json')
with open(fixed_chunk_file_path, "w", encoding="utf-8") as f:
    json.dump(fixed_chunks,f,ensure_ascii=False,indent=2)
with open(recursive_chunk_file_path,"w",encoding="utf-8") as f:
    json.dump(recursive_chunks,f,ensure_ascii=False,indent=2)

## Bước 4: Embedding

In [58]:
%pip install sentence-transformers

In [61]:
embedding_text = r'''from sentence_transformers import SentenceTransformer
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def get_embedding(text):
    return model.encode(text)
'''
python_file_name = "embedding.py"
create_python_file(python_file_name, embedding_text)

Created python file: /content/drive/MyDrive/RAG_Project/RAG_1/src/embedding.py


'/content/drive/MyDrive/RAG_Project/RAG_1/src/embedding.py'

In [97]:
# Check if 'load_pdf' module is already in sys.modules and reload it if necessary
if 'embedding' in sys.modules:
    importlib.reload(sys.modules['embedding'])

%cd /content/drive/MyDrive/RAG_Project/RAG_1/src/
from embedding import get_embedding
query = "Hệ thống IoT là gì?"
vector = get_embedding( query )
print(vector.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/content/drive/MyDrive/RAG_Project/RAG_1/src
(384,)


In [63]:
print(vector)

[-5.03395312e-02  5.43749109e-02 -2.43362179e-03  5.16462103e-02
  4.22097258e-02 -1.14126399e-01  1.11566372e-01  9.36122909e-02
  2.37598382e-02  1.17792804e-02  9.18856170e-03  1.85285248e-02
 -5.35643939e-03 -5.26680704e-03 -6.97510689e-02 -3.96474600e-02
  4.58446294e-02 -9.45805460e-02 -4.29011307e-05 -4.88215312e-02
 -1.05122924e-02  6.02455214e-02  1.19786980e-02 -6.16360307e-02
 -1.54785523e-02  2.41547246e-02  1.52371218e-02 -6.81014732e-02
 -1.55182593e-02 -3.75982746e-02  1.76507495e-02 -7.40955118e-03
  2.36214753e-02 -9.59534757e-03 -9.16503891e-02  4.00776938e-02
  9.52444822e-02 -6.38321713e-02 -3.05422191e-02 -1.39678884e-02
 -3.07266358e-02 -1.15601823e-01 -4.66356799e-02 -3.97226252e-02
  4.38874811e-02  4.86402102e-02 -3.58652100e-02 -1.25064611e-01
 -9.71281305e-02 -5.17367572e-03 -1.51901664e-02 -2.88104750e-02
 -8.25910047e-02  5.99069297e-02  5.43076172e-02 -2.65024602e-02
  4.13384140e-02  1.85833853e-02 -1.98698901e-02 -1.12310657e-02
  8.39229152e-02  2.28334

## Bước 5: Vector hóa toàn bộ chunks

In [98]:
from embedding import get_embedding
fixed_chunks_vector = []
recursive_chunks_vector = []
for chunk in fixed_chunks:
    fixed_chunks_vector.append(get_embedding(chunk))
for chunk in recursive_chunks:
    recursive_chunks_vector.append(get_embedding(chunk))
print(len(fixed_chunks_vector))
print(len(recursive_chunks_vector))

75
74


## Bước 6: Tạo vector store

In [99]:
vector_store_fixed = []
for chunk, vector in zip(fixed_chunks, fixed_chunks_vector):
    vector_store_fixed.append({
        "text": chunk,
        "vector": vector.tolist()
    })
vector_store_recursive = []
for chunk, vector in zip(recursive_chunks, recursive_chunks_vector):
    vector_store_recursive.append({
        "text": chunk,
        "vector": vector.tolist()
    })
print(vector_store_fixed[0])
print(vector_store_recursive[0])

{'text': 'Chương 1. Kiến trúc IoT  \n \n2 \n \nCHƯƠNG 1. KIẾN TRÚC IoT  \n Chương 1 cung cấp cái nhìn tổng quan về IoT, bao gồm các khái niệm cơ \nbản của nó. Chương này cũng bao g ồm các khái ni ệm và ki ến trúc. Ngoài ra, \nchương bao hàm ph ạm vi r ộng nhiều khía c ạnh kỹ thuật cũng như phi k ỹ thuật \nbằng cách trình bày một khuôn khổ bốn lớp đề cập đến các công nghệ cho phép \nthực hiện IoT, các tính chất bắt nguồn từ các hệ thống CNTT-TT hiện đại và cách \nchúng đang hỗ trợ IoT, tiềm năng cũng như các cải tiến mới dựa trên hệ sinh thái \nIoT và cuối cùng là các tác động và thách thức của IoT. \n1.1 Tổng quan về IoT  \nĐầu năm 1926, Nikola Tesla đã hình dung ra m ột “thế giới được kết nối”. \nÔng nói với Tạp chí Colliers trong một cuộc phỏng vấn [84]: \n“Khi không dây được áp dụng một cách hoàn hảo, toàn bộ Trái đất sẽ \nđược chuyển đổi thành một bộ não khổng lồ, trên thực tế, tất cả mọi thứ đều \nlà những hạt của một tổng thể thực sự và nhịp nhàng … và các công cụ mà \nchúng tôi 

## Bước 7: Similar Search

In [68]:
cosin_text = r'''import numpy as np
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
# Tìm top k
def find_top_k(query, vectors, k=3):
    scores = []
    for item in vectors:
        score = cosine_similarity(query, item["vector"])
        scores.append((score, item['text']))
    scores.sort(reverse=True)
    return scores[:k]
'''
python_file_name = "retrieve.py"
create_python_file(python_file_name, cosin_text)
print(f"Create python file: {python_file_name}")

Created python file: /content/drive/MyDrive/RAG_Project/RAG_1/src/retrieve.py
Create python file: retrieve.py


## Bước 9: Ghép context

In [84]:
def concat_context(query_string, top_k=3):

    query_vector = get_embedding(query_string)

    fixed_results = find_top_k(
        query_vector,
        vector_store_fixed,
        top_k
    )

    recursive_results = find_top_k(
        query_vector,
        vector_store_recursive,
        top_k
    )

    print(f"\nQUERY: {query_string}")

    print("\n" + "="*80)
    print("FIXED CHUNKING")
    print("="*80)

    context_fixed = ""

    for rank, (score, chunk) in enumerate(fixed_results, start=1):

        print(f"\nRank {rank}")
        print(f"Score: {score:.4f}")
        print("-"*50)

        print(chunk[:300])

        context_fixed += chunk + "\n\n"

    print("\n" + "="*80)
    print("RECURSIVE CHUNKING")
    print("="*80)

    context_recursive = ""

    for rank, (score, chunk) in enumerate(recursive_results, start=1):

        print(f"\nRank {rank}")
        print(f"Score: {score:.4f}")
        print("-"*50)

        print(chunk[:300])

        context_recursive += chunk + "\n\n"

    return context_fixed, context_recursive

In [102]:
context_fixed, context_recursive = concat_context(
    "IPv6 là gì?"
)


QUERY: IPv6 là gì?

FIXED CHUNKING

Rank 1
Score: 0.7929
--------------------------------------------------
nh. Điều này có 
nghĩa là bất kỳ thiết bị nào triển khai công nghệ yêu cầu lớp thích ứng 
IPv6 phải giao tiếp qua mạng con chỉ sử dụng IPv6.   
Ghi chú: Các cơ chế chuyển tiếp như Ánh xạ địa chỉ và cổng 
sử dụng Phiên dịch (MAP-T55) cho phép chuyển tiếp lưu lượng 
IPv4 qua mạng IPv6. Các k ỹ thuật n

Rank 2
Score: 0.6933
--------------------------------------------------
hiết bị với 
một mạng trong khi chỉ định một địa chỉ IP duy nhất cho định danh và xác định 
vị trí. Với tốc độ phát triển nhanh chóng của Internet sau khi thương mại hóa vào 
những năm 1990, rõ ràng là cần nhiều địa chỉ hơn để kết nối tất cả các thiết bị so 
với người tiền nhiệm của nó - IPv4 - đã c

Rank 3
Score: 0.6241
--------------------------------------------------
 là không trạng thái và về mặt khái niệm thì nó không 
quá phức tạp. Tuy nhiên, một số yếu tố ảnh hưởng đến lượng nén, chẳng hạn như 
việc triể